# A6b Keyword & TF-IDF Baselines — GOLD (CPU, human-verified labels)

Mengevaluasi baseline keyword + TF-IDF terhadap **label human-gold** (`gold.jsonl`)
memakai assignment split leakage-safe yang SAMA dengan silver (review -> split
tidak diubah; hanya label referensi yang diganti dari silver ke gold). Keyword
dihitung ulang terhadap gold; TF-IDF dilatih ulang pada gold train, threshold
di-tune pada gold validation, lalu gold test dievaluasi **sekali**.

Output terpisah dari artefak silver yang dibekukan: `keyword-gold-v1-test-metrics.json`
dan `tfidf-gold-v1-test-metrics.json` dengan `reference_label_type = "human_gold"`.

Ikuti `docs/reproducibility-runbook.md` dan `tools/adjudicator/README.md` sebelum eksekusi.
Prasyarat: `gold.jsonl` sudah di-freeze via `annotation-agreement` + `freeze-gold`.


## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Step 2 — Konfigurasi path & parameter

In [ ]:
# ============================================================
# CONFIGURATION CELL — satu-satunya tempat mengubah parameter.
# ============================================================
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/SIPATURE")
DRIVE_GOLD_DIR = DRIVE_ROOT / "data" / "annotations" / "gold"
DRIVE_SPLIT_DIR = DRIVE_ROOT / "data" / "splits"

PROJECT_DIR = Path("/content/hackathon/ml")
GOLD_DIR = PROJECT_DIR / "data" / "annotations" / "gold"
SPLIT_DIR = PROJECT_DIR / "data" / "splits"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"

DRIVE_METRICS_DIR = DRIVE_ROOT / "metrics"
DRIVE_REPORT_DIR = DRIVE_ROOT / "reports"

GOLD_FILE = "gold.jsonl"
SPLIT_FILES = [
    "train_silver_v1.jsonl",
    "validation_silver_v1.jsonl",
    "test_silver_v1.jsonl",
    "split_manifest_silver_v1.json",
]

print("Drive root:", DRIVE_ROOT)
print("Sumber gold :", DRIVE_GOLD_DIR / GOLD_FILE)
print("Sumber split:", DRIVE_SPLIT_DIR)
print("Artifact dir (lokal):", ARTIFACT_DIR)


## Step 3 — Clone repository dari GitHub

In [ ]:
from google.colab import userdata
import base64
import os
import shutil
import subprocess

token = userdata.get("GITHUB_TOKEN")
assert token, "GITHUB_TOKEN tidak ditemukan di Colab Secrets"

credentials = f"x-access-token:{token}"
authorization = base64.b64encode(credentials.encode()).decode()

repo_dir = "/content/hackathon"
shutil.rmtree(repo_dir, ignore_errors=True)

environment = os.environ.copy()
environment["GIT_CONFIG_COUNT"] = "1"
environment["GIT_CONFIG_KEY_0"] = "http.extraHeader"
environment["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {authorization}"

result = subprocess.run(
    ["git", "clone", "https://github.com/jodypangaribuan/hackathon.git", repo_dir],
    env=environment,
    text=True,
    capture_output=True,
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)

assert result.returncode == 0, "Clone gagal. Periksa izin token GitHub."


## Step 4 — Verifikasi commit terbaru (git log)

In [ ]:
%cd /content/hackathon/ml
!git log --oneline -3


## Step 5 — Install dependencies

In [ ]:
%cd /content/hackathon/ml
!python -m pip install -r requirements-colab.lock.txt
!python -m pip install --no-deps -e .


## Step 6 — Verifikasi versi package

In [ ]:
import numpy
import pandas
import pyarrow
import sklearn
import joblib

print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("PyArrow:", pyarrow.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Joblib:", joblib.__version__)


## Step 7 — Copy gold + split dari Drive ke lokal

In [ ]:
# Salin gold.jsonl + split (train/validation/test + manifest) dari Drive.
import shutil
from pathlib import Path

GOLD_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

gold_source = DRIVE_GOLD_DIR / GOLD_FILE
assert gold_source.is_file(), (
    f"gold.jsonl tidak ditemukan di Drive: {gold_source}\n"
    "Upload gold.jsonl hasil freeze-gold ke SIPATURE/data/annotations/gold/ terlebih dahulu."
)
shutil.copy2(gold_source, GOLD_DIR / GOLD_FILE)
print("Disalin:", GOLD_FILE, "->", GOLD_DIR)

for filename in SPLIT_FILES:
    source = DRIVE_SPLIT_DIR / filename
    assert source.is_file(), f"Split file tidak ditemukan di Drive: {source}"
    shutil.copy2(source, SPLIT_DIR / filename)
    print("Disalin:", filename)


## Step 8 — Import modul sipature_ml

In [ ]:
import sys
from pathlib import Path

source_dir = PROJECT_DIR / "src"

assert source_dir.is_dir(), "Folder source SIPATURE tidak ditemukan."

if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

import sipature_ml

print("Modul SIPATURE berhasil dimuat dari:")
print(sipature_ml.__file__)


## Step 9 — Validasi gold & verifikasi split terkunci

In [ ]:
import json
from pathlib import Path

from sipature_ml.gold_baselines import validate_gold_records
from sipature_ml.manifest import sha256_file

# 1) Validasi gold JSONL (gold tidak punya annotator_id -> validasi label langsung).
validation = validate_gold_records(GOLD_DIR / GOLD_FILE)
print("Gold validation -> records:", validation["records"],
      "| invalid:", validation["invalid_records"])
assert validation["invalid_records"] == 0, f"Gold invalid: {validation['errors']}"

# 2) Verifikasi split manifest terkunci + hash split tidak berubah.
manifest = json.loads((SPLIT_DIR / "split_manifest_silver_v1.json").read_text(encoding="utf-8"))
assert manifest.get("test_is_locked"), "Split manifest tidak terkunci"
for split, output in manifest["outputs"].items():
    actual = sha256_file(SPLIT_DIR / output["path"])
    assert actual == output["sha256"], f"Hash split berubah: {split}"
print("Split locked & hash verifikasi OK.")

# 3) Verifikasi cakupan: review_id gold = review_id split (1320).
gold_ids = {json.loads(line)["review_id"] for line in (GOLD_DIR / GOLD_FILE).read_text().splitlines() if line}
split_ids = set()
for split in ("train", "validation", "test"):
    split_ids |= {json.loads(line)["review_id"] for line in (SPLIT_DIR / f"{split}_silver_v1.jsonl").read_text().splitlines() if line}
print("Gold records:", len(gold_ids), "| Split records:", len(split_ids),
      "| identical:", gold_ids == split_ids)
assert gold_ids == split_ids, "Cakupan review gold != split"


## Step 10 — Jalankan evaluasi gold baselines

In [ ]:
from sipature_ml.gold_baselines import run_gold_baselines

summary = run_gold_baselines(SPLIT_DIR, GOLD_DIR / GOLD_FILE, ARTIFACT_DIR)

print("Keyword Macro F1:", round(summary["keyword_test_macro_f1"], 4))
print("Keyword Micro F1:", round(summary["keyword_test_micro_f1"], 4))
print("TF-IDF  Macro F1:", round(summary["tfidf_test_macro_f1"], 4))
print("TF-IDF  Micro F1:", round(summary["tfidf_test_micro_f1"], 4))
print("TF-IDF representation:", summary["selected_tfidf_representation"])


## Step 11 — Tampilkan perbandingan silver vs gold

In [ ]:
import json
from pathlib import Path

gold_kw = json.loads((ARTIFACT_DIR / "metrics" / "keyword-gold-v1-test-metrics.json").read_text())
gold_tf = json.loads((ARTIFACT_DIR / "metrics" / "tfidf-gold-v1-test-metrics.json").read_text())

def silver_macro(key):
    path = ARTIFACT_DIR / "metrics" / f"{key}-silver-v1-test-metrics.json"
    if not path.is_file():
        return None
    return json.loads(path.read_text())["macro_f1"]

print(f"{'model':<10}{'silver':>10}{'gold':>10}{'delta':>10}")
for key, label, gold in (("keyword", "Keyword", gold_kw["macro_f1"]), ("tfidf", "TF-IDF", gold_tf["macro_f1"])):
    silver = silver_macro(key)
    if silver is not None:
        print(f"{label:<10}{silver:>10.4f}{gold:>10.4f}{gold-silver:>+10.4f}")

print("\nPer-aspect F1 (gold test):")
for aspect, info in summary["per_aspect"].items():
    print(f"  {aspect:<22} kw={info['keyword_f1']:.4f} tfidf={info['tfidf_f1']:.4f} (sup {info['support']})")


## Step 12 — Copy output ke Drive

In [ ]:
# Salin metrics + summary ke Drive (artefak persisten).
import shutil
from pathlib import Path

for local_dir, drive_dir in (
    (ARTIFACT_DIR / "metrics", DRIVE_METRICS_DIR),
    (ARTIFACT_DIR / "reports", DRIVE_REPORT_DIR),
):
    drive_dir.mkdir(parents=True, exist_ok=True)
    for source in sorted(local_dir.glob("*gold-v1*")) + sorted(local_dir.glob("gold_baseline_summary.json")):
        if source.is_file():
            shutil.copy2(source, drive_dir / source.name)
            print(f"Disalin: {source.name} -> {drive_dir}")


## Step 13 — Run summary (hash & metric)

In [ ]:
# ============================================================
# RUN SUMMARY — hash, metric, dan limitations.
# ============================================================
import json
from pathlib import Path
from sipature_ml.manifest import sha256_file

print("REFERENCE LABEL TYPE:", summary["reference_label_type"])
print("SPLIT VERSION        :", summary["split_version"])
print("GOLD SHA256          :", summary["gold_sha256"])
print("Keyword Macro F1     :", round(summary["keyword_test_macro_f1"], 4))
print("TF-IDF  Macro F1     :", round(summary["tfidf_test_macro_f1"], 4))

print("\nOUTPUT METRICS DIR  :", ARTIFACT_DIR / "metrics")
print("OUTPUT REPORTS DIR   :", ARTIFACT_DIR / "reports")

print("\nREMINDER: metric ini adalah agreement terhadap HUMAN-GOLD labels.")
print("Keyword silver F1 0.9768 bersifat circular terhadap silver rules;")
print("gold F1 lebih rendah dan jujur. TF-IDF lebih robust (delta lebih kecil).")
